In [ ]:
import pandas as pd

r_cols = ['anime_id','name','genre','type','episodes','rating','members']
ratings = pd.read_csv('/content/anime.csv', sep=',', usecols=range(7), encoding="ISO-8859-1")

m_cols = ['user_id','anime_id','rating']
animes = pd.read_csv('/content/rating.csv', sep=',', usecols=m_cols, encoding="ISO-8859-1")

# Ensure 'anime_id' columns are of integer type to prevent merge issues
animes['anime_id'] = pd.to_numeric(animes['anime_id'], errors='coerce').fillna(0).astype(int)
ratings['anime_id'] = pd.to_numeric(ratings['anime_id'], errors='coerce').fillna(0).astype(int)

ratings = pd.merge(animes, ratings, on='anime_id')

# Clean column names by stripping whitespace
ratings.columns = ratings.columns.str.strip()

# Rename the merged rating columns
ratings = ratings.rename(columns={'rating_x': 'user_rating', 'rating_y': 'total_rating'})

In [ ]:
ratings.head()

,user_id,anime_id,user_rating,name,genre,type,episodes,total_rating,members
0,1,20,-1.0,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
1,1,24,-1.0,School Rumble,"Comedy, Romance, School, Shounen",TV,26,8.06,178553
2,1,79,-1.0,Shuffle!,"Comedy, Drama, Ecchi, Fantasy, Harem, Magic, R...",TV,24,7.31,158772
3,1,226,-1.0,Elfen Lied,"Action, Drama, Horror, Psychological, Romance,...",TV,13,7.85,623511
4,1,241,-1.0,Girls Bravo: First Season,"Comedy, Ecchi, Fantasy, Harem, Romance, School",TV,11,6.69,84395


### Handling Missing User Ratings (Revisited)

Before creating a sparse matrix, it's essential to properly handle the `-1` values in the `user_rating` column. As discussed, these represent unrated items. Converting them to `NaN` ensures they are not treated as valid ratings and can be implicitly ignored when constructing the sparse matrix.

In [ ]:
import numpy as np

# Convert -1 in 'user_rating' to NaN
ratings['user_rating'] = ratings['user_rating'].replace(-1, np.nan)

# Display the head to show the change and ensure NaNs are present
display(ratings.head())

,user_id,anime_id,user_rating,name,genre,type,episodes,total_rating,members
0,1,20,NaN,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
1,1,24,NaN,School Rumble,"Comedy, Romance, School, Shounen",TV,26,8.06,178553
2,1,79,NaN,Shuffle!,"Comedy, Drama, Ecchi, Fantasy, Harem, Magic, R...",TV,24,7.31,158772
3,1,226,NaN,Elfen Lied,"Action, Drama, Horror, Psychological, Romance,...",TV,13,7.85,623511
4,1,241,NaN,Girls Bravo: First Season,"Comedy, Ecchi, Fantasy, Harem, Romance, School",TV,11,6.69,84395


### Directly Creating a Sparse User-Item Matrix

To avoid the 'out of memory' error, we will directly construct a `scipy.sparse.csr_matrix` from the `ratings` DataFrame. This approach is highly memory-efficient for sparse data by only storing the non-zero (i.e., non-`NaN` ratings) elements. We will also create mappings to easily convert between original user IDs/anime names and their integer indices in the sparse matrix.

In [ ]:
import scipy.sparse as sp

# Filter out NaN ratings before creating the sparse matrix
# This ensures only actual ratings are included
valid_ratings = ratings.dropna(subset=['user_rating'])

# Create mappings for user_id and anime 'name' to integer indices
users = valid_ratings['user_id'].unique()
animes = valid_ratings['name'].unique()

user_to_idx = {user: i for i, user in enumerate(users)}
anime_to_idx = {anime: i for i, anime in enumerate(animes)}
idx_to_anime = {i: anime for i, anime in enumerate(animes)} # For easy lookup later

# Prepare data for sparse matrix
row_indices = valid_ratings['user_id'].map(user_to_idx).values
col_indices = valid_ratings['name'].map(anime_to_idx).values
data = valid_ratings['user_rating'].values

# Create the sparse matrix (CSR format is generally good for row-wise operations)
anime_user_sparse_matrix = sp.csr_matrix(
    (data, (row_indices, col_indices)),
    shape=(len(users), len(animes))
)

print(f"Shape of the sparse matrix: {anime_user_sparse_matrix.shape}")
print(f"Number of non-zero elements: {anime_user_sparse_matrix.nnz}")
print(f"Sparsity: {1 - anime_user_sparse_matrix.nnz / (anime_user_sparse_matrix.shape[0] * anime_user_sparse_matrix.shape[1]):.2%}")

Shape of the sparse matrix: (14748, 8894)
Number of non-zero elements: 1305802
Sparsity: 99.00%


### Creating a User-Item Interaction Matrix

With the `user_rating` column now representing individual user ratings (with unrated items as `NaN`), we can create a user-item interaction matrix using `pivot_table`. This matrix is a fundamental component for many recommendation system algorithms.

In [ ]:
# anime_user_sparse_matrix.head() # This method is not available for sparse matrices

# You can inspect its properties:
print(f"Shape of the sparse matrix: {anime_user_sparse_matrix.shape}")
print(f"Number of non-zero elements: {anime_user_sparse_matrix.nnz}")

# Or to view a small portion (use with caution for very large matrices):
display(anime_user_sparse_matrix[0:5, 0:5].toarray())

Shape of the sparse matrix: (14748, 8894)
Number of non-zero elements: 1305802


array([[10., 10., 10., 10.,  0.],
       [ 0.,  0.,  0.,  0., 10.],
       [ 6.,  0.,  9.,  0., 10.],
       [ 2.,  2.,  1.,  1.,  0.],
       [ 0.,  6.,  8.,  8.,  0.]])

In [ ]:
# Get the original user IDs and anime names for the first 5x5 slice
# We need the inverse mapping for user IDs
idx_to_user = {v: k for k, v in user_to_idx.items()}

sample_user_ids = [idx_to_user[i] for i in range(10)]
sample_anime_names = [idx_to_anime[i] for i in range(10)]

# Convert the 5x5 sparse matrix slice to a dense array and then to a DataFrame
sample_matrix_df = pd.DataFrame(
    anime_user_sparse_matrix[0:10, 0:10].toarray(),
    index=sample_user_ids,
    columns=sample_anime_names
)

display(sample_matrix_df)

,Highschool of the Dead,High School DxD,Sword Art Online,High School DxD New,Kuroko no Basket,Naruto,Shaman King,Slam Dunk,Sen to Chihiro no Kamikakushi,Dragon Ball GT
1,10.0,10.0,10.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0
3,6.0,0.0,9.0,0.0,10.0,8.0,6.0,9.0,10.0,9.0
5,2.0,2.0,1.0,1.0,0.0,6.0,0.0,6.0,8.0,1.0
7,0.0,6.0,8.0,8.0,0.0,0.0,0.0,9.0,0.0,7.0
8,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0
12,6.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0


In [ ]:
naruto_ratings = ratings[ratings['name'] == 'Naruto']
display(naruto_ratings.head())

,user_id,anime_id,user_rating,name,genre,type,episodes,total_rating,members
0,1,20,NaN,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
156,3,20,8.0,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
306,5,20,6.0,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
769,6,20,NaN,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
1162,10,20,NaN,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297


### Building a K-Nearest Neighbors (KNN) Recommendation System

Now that we have our sparse user-item interaction matrix, we can use a K-Nearest Neighbors (KNN) algorithm to find similar items or users. We'll start with an item-based approach, meaning we'll find anime similar to a given anime based on how users have rated them.

In [ ]:
from sklearn.neighbors import NearestNeighbors

# We'll use an item-based collaborative filtering approach.
# This means we want to find items (anime) that are similar to each other.
# For this, we need to transpose our sparse matrix so that anime are rows and users are columns.
# This allows NearestNeighbors to find neighbors among anime based on user ratings.

anime_features = anime_user_sparse_matrix.T

# Initialize the NearestNeighbors model.
# 'metric='cosine'' is often used for recommendation systems as it measures the angle between two vectors,
# which is good for determining similarity regardless of magnitude.
# 'algorithm='brute'' is simple but can be slow for very large datasets; alternatives like 'ball_tree' or 'kd_tree' can be faster.
# 'n_neighbors' specifies how many neighbors to find.
knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20, n_jobs=-1)

# Fit the model to our anime features (transposed sparse matrix)
knn_model.fit(anime_features)

print(f"KNN model fitted with {anime_features.shape[0]} items and {anime_features.shape[1]} features (users).")

KNN model fitted with 8894 items and 14748 features (users).


In [ ]:
import joblib

# Define the filename for the model
model_filename = 'knn_anime_recommender_model.pkl'

# Save the trained model to a file
joblib.dump(knn_model, model_filename)

print(f"KNN model saved to {model_filename}")

KNN model saved to knn_anime_recommender_model.pkl


In [ ]:
import joblib

# Define the filename for the anime features matrix
anime_features_filename = 'anime_features_matrix.pkl'

# Save the anime_features matrix to a file
joblib.dump(anime_features, anime_features_filename)

print(f"Anime features matrix saved to {anime_features_filename}")

Anime features matrix saved to anime_features_matrix.pkl


In [ ]:
# Also, it's very important to save the mappings (anime_to_idx and idx_to_anime)
# as these are needed to translate anime names to indices and vice-versa
# when using the model for recommendations.

joblib.dump(anime_to_idx, 'anime_to_idx_map.pkl')
joblib.dump(idx_to_anime, 'idx_to_anime_map.pkl')

print("Anime index mappings saved to 'anime_to_idx_map.pkl' and 'idx_to_anime_map.pkl'")

Anime index mappings saved to 'anime_to_idx_map.pkl' and 'idx_to_anime_map.pkl'


In [ ]:
# To confirm the model is saved, you can list the files in the current directory
!ls -lh *.pkl

-rw-r--r-- 1 root root  16M Jun  8 05:45 anime_features_matrix.pkl
-rw-r--r-- 1 root root 273K Jun  8 05:45 anime_to_idx_map.pkl
-rw-r--r-- 1 root root 273K Jun  8 05:45 idx_to_anime_map.pkl
-rw-r--r-- 1 root root  15M Jun  8 05:45 knn_anime_recommender_model.pkl


In [ ]:
# Verify all saved files
!ls -lh *.pkl

-rw-r--r-- 1 root root  16M Jun  8 05:45 anime_features_matrix.pkl
-rw-r--r-- 1 root root 273K Jun  8 05:45 anime_to_idx_map.pkl
-rw-r--r-- 1 root root 273K Jun  8 05:45 idx_to_anime_map.pkl
-rw-r--r-- 1 root root  15M Jun  8 05:45 knn_anime_recommender_model.pkl


#### Example: Finding Similar Anime

Let's test our KNN model by finding anime similar to a particular anime. We'll pick 'Naruto' as an example, since you just looked at its ratings.

In [ ]:
# Function to get recommendations
def get_similar_animes(anime_name, model, data, anime_to_idx_map, idx_to_anime_map, n_recommendations=10):
    # Ensure the anime exists in our mapping
    if anime_name not in anime_to_idx_map:
        print(f"Anime '{anime_name}' not found in the dataset.")
        return []

    anime_idx = anime_to_idx_map[anime_name]
    # Get distances and indices of n_recommendations + 1 neighbors (first one is the anime itself)
    distances, indices = model.kneighbors(data[anime_idx], n_neighbors=n_recommendations + 1)

    # Flatten the arrays
    distances = distances.flatten()
    indices = indices.flatten()

    recommendations = []
    for i in range(len(distances)):
        # Skip the first result, as it's the anime itself
        if i == 0:
            continue
        recommendations.append({
            'anime': idx_to_anime_map[indices[i]],
            'similarity': 1 - distances[i] # Convert distance to similarity
        })

    # Sort by similarity in descending order
    recommendations = sorted(recommendations, key=lambda x: x['similarity'], reverse=True)

    return recommendations

# Get recommendations for 'Naruto'
naruto_recommendations = get_similar_animes(
    'Naruto',
    knn_model,
    anime_features,
    anime_to_idx,
    idx_to_anime
)

print(f"Recommendations for 'Naruto':")
for rec in naruto_recommendations:
    print(f"- {rec['anime']} (Similarity: {rec['similarity']:.4f})")

Recommendations for 'Naruto':
- Death Note (Similarity: 0.5557)
- Fullmetal Alchemist (Similarity: 0.4851)
- Code Geass: Hangyaku no Lelouch (Similarity: 0.4795)
- Shingeki no Kyojin (Similarity: 0.4710)
- Fullmetal Alchemist: Brotherhood (Similarity: 0.4673)
- Bleach (Similarity: 0.4652)
- Sword Art Online (Similarity: 0.4628)
- Dragon Ball Z (Similarity: 0.4625)
- Code Geass: Hangyaku no Lelouch R2 (Similarity: 0.4606)
- Elfen Lied (Similarity: 0.4382)


### Evaluating Recommendation Performance: Co-Rating Alignment

To quantitatively assess the quality of our item-item recommendations, we can examine the "co-rating alignment." The idea is that if an anime is a good recommendation for a target anime, then users who rated the target anime highly should also tend to rate the recommended anime highly.

This evaluation strategy involves:
1.  **Selecting a target anime** (e.g., 'Naruto') for which we want to evaluate the recommendations.
2.  **Retrieving its top `N` recommendations** using our `get_similar_animes` function.
3.  **For each recommended anime:**
    *   Identify the set of users who rated *both* the original target anime and the recommended anime.
    *   Calculate the average `user_rating` given to the recommended anime by these overlapping users. A higher average co-rating suggests better alignment with the tastes of users who enjoy the target anime.

This metric helps us understand if the similarities identified by the KNN model translate into actual shared positive sentiment among users.

In [ ]:
# Function to evaluate recommendations based on co-ratings
def evaluate_recommendations_by_co_rating(target_anime_name, recommendations, all_ratings_df):
    print(f"\nEvaluating recommendations for '{target_anime_name}':")

    # Get user_ids who rated the target_anime and ensure their rating is not NaN
    target_anime_user_ratings = all_ratings_df[
        (all_ratings_df['name'] == target_anime_name) &
        (all_ratings_df['user_rating'].notna())
    ]
    target_anime_users = target_anime_user_ratings['user_id'].unique()

    if len(target_anime_users) == 0:
        print(f"No valid user ratings found for '{target_anime_name}'. Cannot evaluate by co-rating.")
        return pd.DataFrame()

    evaluation_results = []
    for rec in recommendations:
        rec_name = rec['anime']
        similarity = rec['similarity']

        # Get ratings for the recommended anime *only from users who also rated the target anime*
        co_ratings = all_ratings_df[
            (all_ratings_df['name'] == rec_name) &
            (all_ratings_df['user_id'].isin(target_anime_users)) &
            (all_ratings_df['user_rating'].notna())
        ]

        if not co_ratings.empty:
            avg_co_rating = co_ratings['user_rating'].mean()
            num_co_raters = len(co_ratings['user_id'].unique())
        else:
            avg_co_rating = np.nan # No overlapping users rated this recommendation
            num_co_raters = 0

        evaluation_results.append({
            'Recommended Anime': rec_name,
            'Similarity (to target)': f'{similarity:.4f}',
            'Avg Co-Rating (from target users)': f'{avg_co_rating:.2f}' if not pd.isna(avg_co_rating) else 'N/A',
            'Num Co-Raters (from target users)': num_co_raters
        })

    # Convert to DataFrame for better display and sort by number of co-raters, then average rating
    eval_df = pd.DataFrame(evaluation_results)
    eval_df['Avg Co-Rating (from target users)'] = pd.to_numeric(eval_df['Avg Co-Rating (from target users)'], errors='coerce')
    eval_df = eval_df.sort_values(by=['Num Co-Raters (from target users)', 'Avg Co-Rating (from target users)'], ascending=[False, False])

    return eval_df

In [ ]:
# Evaluate the recommendations for 'Naruto' using the new function
naruto_eval_df = evaluate_recommendations_by_co_rating(
    'Naruto',
    naruto_recommendations,
    ratings # Use the pre-processed 'ratings' DataFrame
)

# Display the evaluation results
display(naruto_eval_df)


Evaluating recommendations for 'Naruto':


,Recommended Anime,Similarity (to target),Avg Co-Rating (from target users),Num Co-Raters (from target users)
0,Death Note,0.5557,8.89,3312
3,Shingeki no Kyojin,0.4710,8.75,2550
6,Sword Art Online,0.4628,8.09,2518
2,Code Geass: Hangyaku no Lelouch,0.4795,8.97,2454
1,Fullmetal Alchemist,0.4851,8.43,2340
4,Fullmetal Alchemist: Brotherhood,0.4673,9.32,2225
9,Elfen Lied,0.4382,7.99,2220
8,Code Geass: Hangyaku no Lelouch R2,0.4606,9.13,2214
7,Dragon Ball Z,0.4625,8.32,1787
5,Bleach,0.4652,8.19,1568


### Alternative Recommendation System: Singular Value Decomposition (SVD)

Singular Value Decomposition (SVD) is a matrix factorization technique widely used in recommender systems. It works by decomposing the user-item interaction matrix into three lower-dimensional matrices, which can then be used to predict missing ratings (i.e., user preferences for unrated items).

In [ ]:
import sys
!{sys.executable} -m pip install scikit-surprise

from surprise import Dataset, Reader
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# We need to create a `Reader` object for the Surprise library. This tells Surprise
# about the rating scale. Our user ratings are on a scale of 1 to 10.
reader = Reader(rating_scale=(1, 10))

# Load the dataset from the 'ratings' DataFrame. We only need user_id, anime_id, and user_rating.
# Note: We need to use the original anime_id for Surprise, not the 'name' or the sparse matrix indices.
surprise_data = Dataset.load_from_df(ratings[['user_id', 'anime_id', 'user_rating']].dropna(), reader)

# Split the data into training and testing sets
trainset, testset = train_test_split(surprise_data, test_size=0.25, random_state=42)

print(f"Number of users in trainset: {trainset.n_users}")
print(f"Number of items in trainset: {trainset.n_items}")
print(f"Number of ratings in trainset: {trainset.n_ratings}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.1 MB/s eta 0:00:00
Number of users in trainset: 14539
Number of items in trainset: 8564
Number of ratings in trainset: 979351


In [ ]:
# Initialize and train the SVD model
# n_epochs: number of iteration of the SGD procedure
# lr_all: learning rate for all parameters
# reg_all: regularization term for all parameters
svd_model = SVD(n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd_model.fit(trainset)

print("SVD model training complete.")

SVD model training complete.


In [ ]:
# Make predictions on the testset
predictions = svd_model.test(testset)

# Evaluate the model using RMSE (Root Mean Squared Error)
print("Model Evaluation:")
accuracy.rmse(predictions, verbose=True)

# Example of predicting a rating for a specific user and anime
# First, we need to map anime_id back to its name for display
# We'll use the original 'ratings' DataFrame to get anime names

def get_anime_name_from_id(anime_id, df):
    return df[df['anime_id'] == anime_id]['name'].iloc[0] if not df[df['anime_id'] == anime_id].empty else "Unknown Anime"

user_id_to_predict = 1
anime_id_to_predict = 20 # Naruto

# Predict the rating
predicted_rating = svd_model.predict(user_id_to_predict, anime_id_to_predict).est
print(f"\nPredicted rating for User {user_id_to_predict} on '{get_anime_name_from_id(anime_id_to_predict, ratings)}' (ID: {anime_id_to_predict}): {predicted_rating:.2f}")

# You can also get top N recommendations for a user
def get_top_n_recommendations_svd(user_id, model, original_ratings_df, n=10):
    # Get all anime that the user has not rated
    user_rated_animes = original_ratings_df[original_ratings_df['user_id'] == user_id]['anime_id'].unique()
    all_anime_ids = original_ratings_df['anime_id'].unique()
    animes_to_predict = [anime_id for anime_id in all_anime_ids if anime_id not in user_rated_animes]

    predictions = []
    for anime_id in animes_to_predict:
        predicted_rating = model.predict(user_id, anime_id).est
        predictions.append({'anime_id': anime_id, 'predicted_rating': predicted_rating})

    # Sort predictions by predicted rating
    predictions.sort(key=lambda x: x['predicted_rating'], reverse=True)

    # Get top N and include anime names
    top_n = []
    for p in predictions[:n]:
        anime_name = get_anime_name_from_id(p['anime_id'], original_ratings_df)
        top_n.append({'anime': anime_name, 'predicted_rating': p['predicted_rating']})
    return top_n

print(f"\nTop 10 recommendations for User {user_id_to_predict}:")
top_recommendations = get_top_n_recommendations_svd(user_id_to_predict, svd_model, ratings, n=10)
for rec in top_recommendations:
    print(f"- {rec['anime']} (Predicted Rating: {rec['predicted_rating']:.2f})")

Model Evaluation:
RMSE: 1.1819

Predicted rating for User 1 on 'Naruto' (ID: 20): 8.93

Top 10 recommendations for User 1:
- Code Geass: Hangyaku no Lelouch (Predicted Rating: 10.00)
- Code Geass: Hangyaku no Lelouch R2 (Predicted Rating: 10.00)
- Death Note (Predicted Rating: 9.99)
- Mirai Nikki (TV) (Predicted Rating: 9.94)
- Gintama Movie: Kanketsu-hen - Yorozuya yo Eien Nare (Predicted Rating: 9.88)
- Fairy Tail (Predicted Rating: 9.87)
- Gintama (Predicted Rating: 9.84)
- GintamaÂ° (Predicted Rating: 9.77)
- Rainbow: Nisha Rokubou no Shichinin (Predicted Rating: 9.77)
- Clannad: After Story (Predicted Rating: 9.75)


This SVD model provides an alternative way to generate recommendations based on matrix factorization. The RMSE metric gives an idea of how well the model predicts user ratings. The top N recommendations show anime that a user is predicted to like the most, based on the model's understanding of user preferences and item characteristics.

### SVD Model Summary and Results

We successfully implemented a Singular Value Decomposition (SVD) model using the Surprise library. The model was trained on 75% of the user-anime rating data, and its performance was evaluated on the remaining 25%.

**Key takeaways:**

*   **RMSE:** The model achieved a Root Mean Squared Error (RMSE) of approximately **1.1819**. RMSE measures the average magnitude of the errors between predicted and actual ratings. A lower RMSE indicates a better fit of the model to the data.
*   **Prediction Example:** For User 1 and Anime ID 20 ('Naruto'), the model predicted a rating of **8.93**.
*   **Top 10 Recommendations:** The SVD model also generated a list of top 10 anime recommendations for a given user based on their predicted ratings for unrated items. These recommendations are based on the latent factors learned by the SVD algorithm, representing underlying preferences and item characteristics.

SVD offers a powerful alternative to neighborhood-based methods like KNN, especially in handling data sparsity and identifying latent features that drive user preferences.

### Understanding the Difference: Item-Based KNN vs. SVD (Matrix Factorization)

Your observation about the `user_id` argument is insightful and points to a fundamental difference between our KNN implementation and the SVD model:

*   **Item-Based KNN (as implemented here):**
    *   This approach is focused on **item-to-item similarity**. It looks at how *all users* have rated items to determine which items are similar to each other. For example, if many users who liked 'Naruto' also liked 'Death Note', then 'Death Note' is considered similar to 'Naruto'.
    *   When you ask for recommendations for 'Naruto', the model simply finds anime that are similar to 'Naruto' based on this collective user behavior. It does *not* filter or tailor these recommendations based on a *specific user's* past ratings or preferences.
    *   The output you saw for 'Naruto' (`naruto_recommendations`) is a list of anime that are generally similar to 'Naruto' across the entire dataset of user ratings.

*   **SVD (Matrix Factorization):**
    *   SVD, on the other hand, is a model that **learns user preferences and item characteristics (latent factors)**. It predicts how a *specific user* would rate an *unrated item*.
    *   Therefore, its `get_top_n_recommendations_svd` function *must* take a `user_id` as an argument, because the recommendations are personalized and unique to that individual user. The model iterates through all unrated items for that user and predicts their rating, then recommends the highest-rated ones.

In summary, the KNN model identifies general item similarities, while the SVD model provides personalized predictions and recommendations for individual users. Both are valuable, but they answer slightly different questions in recommendation systems.

### Comparing KNN and SVD Recommendations

Let's put the recommendations side-by-side to highlight the differences. We'll use the recommendations previously generated:

*   **KNN Recommendations for 'Naruto':** These are anime similar to Naruto based on overall user ratings.
*   **SVD Recommendations for User 1:** These are anime that User 1 is predicted to like the most.

In [ ]:
print("--- KNN Recommendations for 'Naruto' ---")
for rec in naruto_recommendations:
    print(f"- {rec['anime']} (Similarity: {rec['similarity']:.4f})")

print("\n--- SVD Top 10 Recommendations for User 1 ---")
# The variable `top_recommendations` from the SVD cell already holds this data
for rec in top_recommendations:
    print(f"- {rec['anime']} (Predicted Rating: {rec['predicted_rating']:.2f})")

--- KNN Recommendations for 'Naruto' ---
- Death Note (Similarity: 0.5557)
- Fullmetal Alchemist (Similarity: 0.4851)
- Code Geass: Hangyaku no Lelouch (Similarity: 0.4795)
- Shingeki no Kyojin (Similarity: 0.4710)
- Fullmetal Alchemist: Brotherhood (Similarity: 0.4673)
- Bleach (Similarity: 0.4652)
- Sword Art Online (Similarity: 0.4628)
- Dragon Ball Z (Similarity: 0.4625)
- Code Geass: Hangyaku no Lelouch R2 (Similarity: 0.4606)
- Elfen Lied (Similarity: 0.4382)

--- SVD Top 10 Recommendations for User 1 ---
- Code Geass: Hangyaku no Lelouch (Predicted Rating: 10.00)
- Code Geass: Hangyaku no Lelouch R2 (Predicted Rating: 10.00)
- Death Note (Predicted Rating: 9.99)
- Mirai Nikki (TV) (Predicted Rating: 9.94)
- Gintama Movie: Kanketsu-hen - Yorozuya yo Eien Nare (Predicted Rating: 9.88)
- Fairy Tail (Predicted Rating: 9.87)
- Gintama (Predicted Rating: 9.84)
- GintamaÂ° (Predicted Rating: 9.77)
- Rainbow: Nisha Rokubou no Shichinin (Predicted Rating: 9.77)
- Clannad: After Story (Pr

### Can SVD Perform Item-to-Item Similarity like KNN?

Your question about whether SVD can perform KNN's task (finding similar items) is insightful. Here's a breakdown:

**SVD's Primary Strength: Personalized Rating Prediction**

As we've seen, SVD excels at predicting *how a specific user would rate a specific item*. It does this by decomposing the user-item interaction matrix into lower-dimensional "latent factor" matrices for both users and items. These latent factors essentially capture hidden characteristics or preferences (e.g., a user's preference for 'action-packed fantasy anime' and an anime's characteristic of being 'action-packed fantasy').

When we ask SVD for recommendations for User 1, it uses User 1's latent factor vector and combines it with the latent factor vectors of all unrated anime to predict a rating for each, then recommends the highest-rated ones. This makes it inherently **user-centric** and **predictive**.

**Can SVD be Used for Item-to-Item Similarity? Yes, Indirectly!**

While not its most direct application, you absolutely *can* derive item-to-item similarity using an SVD model. Here's how:

1.  **Latent Item Vectors:** After SVD training, each item in your dataset (each anime) will have a corresponding latent factor vector (a row or column in one of the decomposed matrices). This vector represents the anime's characteristics in the latent space.
2.  **Calculate Similarity:** To find anime similar to 'Naruto', you would take 'Naruto's latent factor vector and calculate its cosine similarity (or another distance metric) with the latent factor vectors of all other anime.

**Technical and Conceptual Differences from KNN:**

*   **Nature of Similarity:**
    *   **KNN:** Directly calculates similarity based on **explicit co-ratings** from users. If many users rated both A and B similarly, they are similar. This is often more intuitive.
    *   **SVD (Latent):** Calculates similarity based on **latent features** that the model has *learned*. These features are not directly observable like genres or tags, but rather abstract representations of user preferences and item characteristics. This can sometimes capture more nuanced relationships.
*   **Input Data:**
    *   **KNN:** Operates directly on the sparse user-item matrix (or its transpose).
    *   **SVD:** Learns from the `Dataset` object (which typically contains the user, item, and rating triplets) to build a predictive model.
*   **Computational Efficiency:** For very large datasets, calculating similarities in a lower-dimensional latent space (from SVD) can sometimes be more efficient than calculating distances in the high-dimensional original item-feature space for KNN.
*   **Purpose:** If your primary goal is simply to tell a user, "People who liked X also liked Y," then a straightforward item-based KNN is often simpler, more transparent, and computationally less demanding. If your goal is *personalized predictions* and handling extreme data sparsity, SVD is generally preferred.

**In summary:** While SVD's core task is personalized rating prediction, its learned latent factor representations are rich enough to also infer item-to-item similarities. It's a matter of choosing the right tool for the specific job and understanding the underlying mechanism of similarity. We could implement a function to find similar items based on SVD's latent factors, but it would involve extracting those factors and then calculating their similarities, which is a different process than the direct neighbor search in our current KNN implementation.

### Deriving Item-to-Item Similarity from SVD Latent Factors

As discussed, we can extract the item latent factors from our trained SVD model and use them to calculate item-to-item similarity. Each anime will be represented by a vector in a lower-dimensional space, and we can find similar anime by looking for items with similar vectors.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# The SVD model stores item latent factors in `svd_model.qi`
# These are typically in the order of `surprise`'s internal item IDs.
# We need to map our original anime_ids to surprise's internal item IDs.

# First, create a mapping from original anime_id to surprise's internal item index
# This can be done using the `trainset`'s `to_inner_iid` method.

# Get all unique anime_ids from our original ratings DataFrame
all_anime_ids_original = ratings['anime_id'].unique()

# Create a dictionary to store the inner item ID for each original anime_id
anime_id_to_inner_id = {}
inner_id_to_anime_id = {}
for anime_id in all_anime_ids_original:
    try:
        inner_id = trainset.to_inner_iid(anime_id)
        anime_id_to_inner_id[anime_id] = inner_id
        inner_id_to_anime_id[inner_id] = anime_id
    except ValueError: # Anime not in trainset, ignore
        continue

# Create a mapping from surprise's inner item ID to anime name
inner_id_to_anime_name = {inner_id: get_anime_name_from_id(anime_id, ratings) for inner_id, anime_id in inner_id_to_anime_id.items()}

# Extract item latent factors (Q matrix in Surprise terminology)
# This matrix has shape (n_items, n_factors)
item_latent_factors = svd_model.qi

# Function to get similar animes based on SVD latent factors
def get_similar_animes_svd(target_anime_name, svd_model, item_latent_factors, anime_name_to_original_id_map, inner_id_to_anime_name_map, n_recommendations=10):
    # Get the original anime_id for the target anime name
    target_original_id = anime_name_to_original_id_map.get(target_anime_name)
    if target_original_id is None:
        print(f"Anime '{target_anime_name}' not found in original anime_id map.")
        return []

    # Get the inner item ID for the target anime
    target_inner_id = anime_id_to_inner_id.get(target_original_id)
    if target_inner_id is None:
        print(f"Anime '{target_anime_name}' (ID: {target_original_id}) not found in SVD trainset.")
        return []

    # Get the latent factor vector for the target anime
    target_latent_vector = item_latent_factors[target_inner_id].reshape(1, -1)

    similarities = []
    for inner_item_id in range(item_latent_factors.shape[0]):
        if inner_item_id == target_inner_id: # Skip the item itself
            continue

        candidate_latent_vector = item_latent_factors[inner_item_id].reshape(1, -1)
        similarity = cosine_similarity(target_latent_vector, candidate_latent_vector)[0][0]

        # Map inner_item_id back to anime name for display
        anime_name = inner_id_to_anime_name_map.get(inner_item_id, "Unknown Anime")

        similarities.append({
            'anime': anime_name,
            'similarity': similarity
        })

    # Sort by similarity in descending order
    similarities.sort(key=lambda x: x['similarity'], reverse=True)

    return similarities[:n_recommendations]

# Create a reverse map from anime name to original anime_id for the SVD similarity function
anime_name_to_original_id = {name: aid for aid, name in ratings[['anime_id', 'name']].drop_duplicates().set_index('anime_id')['name'].items()}

# Get SVD-based recommendations for 'Naruto'
naruto_svd_similar_animes = get_similar_animes_svd(
    'Naruto',
    svd_model,
    item_latent_factors,
    anime_name_to_original_id,
    inner_id_to_anime_name
)

print("--- SVD Latent Factor Similar Animes for 'Naruto' ---")
for rec in naruto_svd_similar_animes:
    print(f"- {rec['anime']} (Similarity: {rec['similarity']:.4f})")

--- SVD Latent Factor Similar Animes for 'Naruto' ---
- Naruto Narutimate Hero 3: Tsuini Gekitotsu! Jounin vs. Genin!! Musabetsu Dairansen taikai Kaisai!! (Similarity: 0.4252)
- Bleach (Similarity: 0.4058)
- Baguda-jou no Touzoku (Similarity: 0.3952)
- Naruto: Takigakure no Shitou - Ore ga Eiyuu Dattebayo! (Similarity: 0.3882)
- Naruto Movie 1: Dai Katsugeki!! Yuki Hime Shinobu Houjou Dattebayo! (Similarity: 0.3811)
- Naruto: Akaki Yotsuba no Clover wo Sagase (Similarity: 0.3713)
- InuYasha: Guren no Houraijima (Similarity: 0.3540)
- Naruto: Shippuuden Movie 2 - Kizuna (Similarity: 0.3491)
- Bleach Movie 3: Fade to Black - Kimi no Na wo Yobu (Similarity: 0.3453)
- Ookii 1 Nensei to Chiisana 2 Nensei (Similarity: 0.3404)


### Direct Comparison: KNN vs. SVD Item Similarity for 'Naruto'

Now, let's compare the anime similar to 'Naruto' as found by:

1.  **Item-Based KNN:** Based on explicit co-rating patterns from users.
2.  **SVD Latent Factors:** Based on the abstract, learned characteristics of anime from the SVD model.

In [ ]:
print("--- KNN Similar Animes for 'Naruto' ---")
for rec in naruto_recommendations:
    print(f"- {rec['anime']} (Similarity: {rec['similarity']:.4f})")

print("\n--- SVD Latent Factor Similar Animes for 'Naruto' ---")
for rec in naruto_svd_similar_animes:
    print(f"- {rec['anime']} (Similarity: {rec['similarity']:.4f})")

--- KNN Similar Animes for 'Naruto' ---
- Death Note (Similarity: 0.5557)
- Fullmetal Alchemist (Similarity: 0.4851)
- Code Geass: Hangyaku no Lelouch (Similarity: 0.4795)
- Shingeki no Kyojin (Similarity: 0.4710)
- Fullmetal Alchemist: Brotherhood (Similarity: 0.4673)
- Bleach (Similarity: 0.4652)
- Sword Art Online (Similarity: 0.4628)
- Dragon Ball Z (Similarity: 0.4625)
- Code Geass: Hangyaku no Lelouch R2 (Similarity: 0.4606)
- Elfen Lied (Similarity: 0.4382)

--- SVD Latent Factor Similar Animes for 'Naruto' ---
- Naruto Narutimate Hero 3: Tsuini Gekitotsu! Jounin vs. Genin!! Musabetsu Dairansen taikai Kaisai!! (Similarity: 0.4252)
- Bleach (Similarity: 0.4058)
- Baguda-jou no Touzoku (Similarity: 0.3952)
- Naruto: Takigakure no Shitou - Ore ga Eiyuu Dattebayo! (Similarity: 0.3882)
- Naruto Movie 1: Dai Katsugeki!! Yuki Hime Shinobu Houjou Dattebayo! (Similarity: 0.3811)
- Naruto: Akaki Yotsuba no Clover wo Sagase (Similarity: 0.3713)
- InuYasha: Guren no Houraijima (Similarity: 

### Accessing Public S3 Data from Colab

To read data directly from a public Amazon S3 bucket, we need to install the `s3fs` library. This library provides a Pythonic file system interface to S3, allowing `pandas` to open files stored there as if they were local.

### Securely Accessing Private S3 Buckets from Colab

Since making S3 buckets public carries risks, a more secure approach is to use AWS Identity and Access Management (IAM) user credentials. These credentials (Access Key ID and Secret Access Key) grant your Colab environment programmatic access to your private S3 bucket.

**Steps to set up IAM credentials:**

1.  **Create an IAM User in AWS:**
    *   Go to the AWS IAM Console.
    *   Navigate to **Users** and click **Add users**.
    *   Give the user a name (e.g., `colab-s3-access`).
    *   Select **Access key - Programmatic access** as the credential type.
    *   Proceed to **Permissions**. Attach a policy like `AmazonS3ReadOnlyAccess` for read-only access, or create a custom policy with `s3:GetObject` specifically for your bucket (`ml-data-bucket-tutorial-2026`).
    *   Complete the user creation. **Crucially, save the Access Key ID and Secret Access Key displayed on the final screen.** These will only be shown once.

2.  **Store Credentials in Colab Secrets:**
    *   In Google Colab, click the **'🔑 Secrets'** icon on the left sidebar.
    *   Click **'+ New secret'**.
    *   Add two secrets:
        *   **Name:** `AWS_ACCESS_KEY_ID`, **Value:** `your_aws_access_key_id` (from IAM)
        *   **Name:** `AWS_SECRET_ACCESS_KEY`, **Value:** `your_aws_secret_access_key` (from IAM)
    *   Ensure the 'Notebook access' toggle is enabled for these secrets.

3.  **Access S3 from Colab using `s3fs`:**
    *   The `s3fs` library (which `pandas` uses under the hood for S3) will automatically pick up AWS credentials from environment variables. We'll load them from Colab secrets and set them as environment variables.

In [ ]:
# Install s3fs to enable pandas to read from S3
!pip install s3fs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 90.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.4.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.


This code block demonstrates how to load a CSV file from a public S3 bucket. If you have sensitive data, remember to keep your S3 buckets private and use authentication methods (like AWS credentials) for access.

In [ ]:
from google.colab import userdata
import os
import pandas as pd

# Retrieve credentials from Colab Secrets
# Ensure these secret names match what you set in the Colab Secrets panel
aws_access_key_id = userdata.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY')

# Set them as environment variables so s3fs can pick them up
os.environ['AWS_ACCESS_KEY_ID'] = aws_access_key_id
os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret_access_key

# Now, you can access your private S3 bucket.
# Replace with the actual S3 URI to your anime.csv file in your private bucket.
# For example, if your bucket is 'ml-data-bucket-tutorial-2026' and the file is 'anime_dataset/anime.csv'
private_s3_uri = 's3://ml-data-bucket-tutorial-2026/anime_dataset/anime.csv'

try:
    # Read the CSV directly from S3 using the configured credentials
    s3_private_df = pd.read_csv(private_s3_uri)
    print(f"Successfully loaded data from private S3 bucket: {private_s3_uri}")
    display(s3_private_df.head())
except Exception as e:
    print(f"Error loading data from private S3: {e}")
    print("Please ensure your IAM user has the correct permissions and the bucket is NOT public.")


SecretNotFoundError: Secret AWS_ACCESS_KEY_ID does not exist.